# Case Study 3: Deep Hierarchies and Proximal Dominance

**Circulatory Fidelity v1.1: The Proximal Dominance Principle**

This notebook demonstrates the **Proximal Dominance Principle**: in deep hierarchies with fully factorized MFVI, only proximal-layer coupling determines inference quality.

---

## Key Findings

- **Proximal coupling alone**: Up to 40× MSE degradation
- **Distal coupling alone**: Exactly 1.0× (zero degradation, mathematically guaranteed)
- **Combined**: Distal coupling amplifies proximal failure by 1.01–2.26×

**Scope**: This principle applies specifically to fully factorized mean-field approximations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import rankdata, norm
from dataclasses import dataclass
from typing import Tuple, NamedTuple

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11
np.random.seed(42)

## Three-Layer SVF Model

$$
\begin{align}
z^{(3)}_t &= z^{(3)}_{t-1} + \varepsilon_3 \quad \text{[distal layer]}\\
\log\sigma^{(2)}_t &= \kappa_{32} \cdot z^{(3)}_t + \omega_2 \quad \text{[distal coupling]}\\
z^{(2)}_t &= z^{(2)}_{t-1} + \varepsilon_2(\sigma^{(2)}_t) \quad \text{[middle layer]}\\
\log\sigma^{(1)}_t &= \kappa_{21} \cdot z^{(2)}_t + \omega_1 \quad \text{[proximal coupling]}\\
z^{(1)}_t &= z^{(1)}_{t-1} + \varepsilon_1(\sigma^{(1)}_t) \quad \text{[proximal layer]}\\
y_t &= z^{(1)}_t + \varepsilon_y \quad \text{[observation]}
\end{align}
$$

Key parameters:
- $\kappa_{21}$: **Proximal coupling** (middle → proximal)
- $\kappa_{32}$: **Distal coupling** (distal → middle)

In [ ]:
def inference_coupling(x: np.ndarray, y: np.ndarray) -> Tuple[float, float]:
    """Copula-based IC estimation."""
    x = np.asarray(x).flatten()
    y = np.asarray(y).flatten()
    n = len(x)
    
    u = (rankdata(x) - 0.5) / n
    v = (rankdata(y) - 0.5) / n
    z_x = norm.ppf(u)
    z_y = norm.ppf(v)
    rho = np.corrcoef(z_x, z_y)[0, 1]
    
    ic = np.abs(rho)
    se = 1.0 / np.sqrt(n - 3) if n > 3 else np.nan
    
    return ic, se

In [ ]:
@dataclass
class ThreeLayerParams:
    """Three-layer SVF parameters."""
    kappa_32: float = 0.0    # Distal coupling
    kappa_21: float = 0.0    # Proximal coupling
    omega_1: float = -0.5    # Proximal baseline log-volatility
    omega_2: float = -0.5    # Middle baseline log-volatility
    sigma_3: float = 0.3     # Distal noise
    sigma_obs: float = 0.5   # Observation noise

class ThreeLayerSimulation(NamedTuple):
    z3: np.ndarray  # Distal
    z2: np.ndarray  # Middle
    z1: np.ndarray  # Proximal
    y: np.ndarray   # Observations
    vol2: np.ndarray
    vol1: np.ndarray
    params: ThreeLayerParams

def simulate_three_layer(params: ThreeLayerParams, T: int = 300, 
                        seed: int = None) -> ThreeLayerSimulation:
    """Simulate three-layer hierarchy."""
    if seed is not None:
        np.random.seed(seed)
    
    z3 = np.zeros(T)
    z2 = np.zeros(T)
    z1 = np.zeros(T)
    y = np.zeros(T)
    vol2 = np.zeros(T)
    vol1 = np.zeros(T)
    
    for t in range(1, T):
        # Distal layer
        z3[t] = z3[t-1] + np.random.normal(0, params.sigma_3)
        
        # Middle layer (modulated by distal)
        log_vol2 = np.clip(params.kappa_32 * z3[t] + params.omega_2, -3, 3)
        vol2[t] = np.exp(log_vol2 / 2)
        z2[t] = z2[t-1] + np.random.normal(0, vol2[t])
        
        # Proximal layer (modulated by middle)
        log_vol1 = np.clip(params.kappa_21 * z2[t] + params.omega_1, -3, 3)
        vol1[t] = np.exp(log_vol1 / 2)
        z1[t] = z1[t-1] + np.random.normal(0, vol1[t])
        
        # Observation
        y[t] = z1[t] + np.random.normal(0, params.sigma_obs)
    
    return ThreeLayerSimulation(z3=z3, z2=z2, z1=z1, y=y, 
                                vol2=vol2, vol1=vol1, params=params)

## Inference Methods

In [ ]:
def mf_filter(sim: ThreeLayerSimulation) -> Tuple[np.ndarray, float]:
    """Mean-field Kalman: assumes constant volatility."""
    T = len(sim.y)
    avg_vol = np.exp(sim.params.omega_1 / 2)
    
    z1_est = np.zeros(T)
    var_est = np.ones(T)
    
    for t in range(1, T):
        pred_var = var_est[t-1] + avg_vol**2
        obs_var = sim.params.sigma_obs**2
        K = pred_var / (pred_var + obs_var)
        z1_est[t] = z1_est[t-1] + K * (sim.y[t] - z1_est[t-1])
        var_est[t] = (1 - K) * pred_var
    
    mse = np.mean((z1_est - sim.z1)**2)
    return z1_est, mse

def oracle_filter(sim: ThreeLayerSimulation) -> Tuple[np.ndarray, float]:
    """Oracle Kalman: knows true proximal volatility."""
    T = len(sim.y)
    
    z1_est = np.zeros(T)
    var_est = np.ones(T)
    
    for t in range(1, T):
        pred_var = var_est[t-1] + sim.vol1[t]**2
        obs_var = sim.params.sigma_obs**2
        K = pred_var / (pred_var + obs_var)
        z1_est[t] = z1_est[t-1] + K * (sim.y[t] - z1_est[t-1])
        var_est[t] = (1 - K) * pred_var
    
    mse = np.mean((z1_est - sim.z1)**2)
    return z1_est, mse

## Demonstration: Proximal vs Distal Coupling

In [ ]:
configurations = [
    {'name': 'No coupling', 'kappa_32': 0.0, 'kappa_21': 0.0},
    {'name': 'Distal only', 'kappa_32': 1.5, 'kappa_21': 0.0},
    {'name': 'Proximal only', 'kappa_32': 0.0, 'kappa_21': 1.5},
    {'name': 'Both', 'kappa_32': 1.5, 'kappa_21': 1.5},
]

results = []
for config in configurations:
    params = ThreeLayerParams(kappa_32=config['kappa_32'], kappa_21=config['kappa_21'])
    
    mse_ratios = []
    for rep in range(50):
        sim = simulate_three_layer(params, T=300)
        _, mf_mse = mf_filter(sim)
        _, oracle_mse = oracle_filter(sim)
        mse_ratios.append(mf_mse / max(oracle_mse, 1e-10))
    
    results.append({
        'name': config['name'],
        'kappa_32': config['kappa_32'],
        'kappa_21': config['kappa_21'],
        'mse_ratio_mean': np.mean(mse_ratios),
        'mse_ratio_std': np.std(mse_ratios)
    })

df_results = pd.DataFrame(results)
print("Proximal Dominance Demonstration:")
print(df_results.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(df_results))
bars = ax.bar(x, df_results['mse_ratio_mean'], yerr=df_results['mse_ratio_std'],
              capsize=5, color=['green', 'blue', 'red', 'purple'], alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(df_results['name'])
ax.set_ylabel('MSE Ratio (MF / Oracle)')
ax.set_title('Proximal Dominance Principle\nDistal coupling alone causes NO degradation')
ax.axhline(1.0, color='gray', linestyle='--', label='No degradation')

# Annotate
for i, row in df_results.iterrows():
    ax.text(i, row['mse_ratio_mean'] + row['mse_ratio_std'] + 1, 
           f"{row['mse_ratio_mean']:.1f}×", ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## Why Distal-Only Degradation is Exactly 1.0

This is **not** a numerical approximation—it's an **algebraic guarantee**:

When $\kappa_{21} = 0$:
$$\log \sigma_1^2(t) = \kappa_{21} \cdot z_2(t) + \omega_1 = 0 \cdot z_2(t) + \omega_1 = \omega_1 \quad \text{(constant)}$$

The proximal volatility becomes **time-invariant**, regardless of what happens in distal layers.

Mean-field inference assumes constant volatility—which is **exactly correct** when $\kappa_{21} = 0$.

## Validation Against Manuscript Data

In [ ]:
try:
    df = pd.read_csv('../data/three_layer_validation.csv')
    print(f"Loaded validation data: {len(df)} simulations")
    
    # Summary by configuration
    summary = df.groupby(['kappa_32', 'kappa_21']).agg({
        'ic_32': 'mean',
        'ic_21': 'mean', 
        'mse_ratio': ['mean', 'std', 'count']
    }).round(3)
    print("\nSummary by coupling configuration:")
    print(summary)
    
except FileNotFoundError:
    print("Validation data not found.")
    df = None

In [ ]:
if df is not None:
    # Create heatmap
    pivot = df.groupby(['kappa_32', 'kappa_21'])['mse_ratio'].mean().unstack()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(pivot.values, cmap='Reds', aspect='auto')
    
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_yticks(range(len(pivot.index)))
    ax.set_xticklabels([f'{x:.1f}' for x in pivot.columns])
    ax.set_yticklabels([f'{x:.1f}' for x in pivot.index])
    ax.set_xlabel('Proximal coupling κ₂₁')
    ax.set_ylabel('Distal coupling κ₃₂')
    ax.set_title('MSE Ratio by Coupling Configuration\n(Note: Vertical axis has no effect when proximal=0)')
    
    plt.colorbar(im, ax=ax, label='MSE Ratio')
    
    # Annotate
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            ax.text(j, i, f'{val:.1f}', ha='center', va='center', 
                   color='white' if val > 20 else 'black')
    
    plt.tight_layout()
    plt.show()

## Key Findings: The Proximal Dominance Principle

For fully factorized MFVI in deep hierarchies:

1. **Proximal coupling is necessary**: No proximal coupling → no degradation (MSE = 1.0× exactly)
2. **Proximal coupling is sufficient**: Proximal coupling alone causes substantial degradation
3. **Distal coupling amplifies but cannot cause**: Distal adds 1.01–2.26× multiplier to existing proximal failure

### Practical Implication

For $L$-layer models, diagnostic complexity reduces from $O(L^2)$ pairwise checks to $O(1)$: **only check the proximal layer IC**.

### Scope Limitation

This principle applies specifically to **fully factorized** mean-field families. Structured variational families (block mean-field, normalizing flows) may propagate distal information more effectively.

---

*Notebook aligned with Circulatory Fidelity v1.1 manuscript*